In [1]:
import wbgapi as wb
import pandas as pd

In [2]:
df_exp = wb.data.DataFrame("NE.EXP.GNFS.CD", time=range(2010, 2025), labels=True, skipAggs=True)
df_imp= wb.data.DataFrame("NE.IMP.GNFS.CD", time=range(2010, 2025), labels=True, skipAggs=True)
PIB_pays = wb.data.DataFrame("NY.GDP.MKTP.CD", time=range(2010, 2025), labels=True)


In [3]:
PIB_world = PIB_pays[PIB_pays["Country"] == "World"]

In [4]:
df_exp.rename(
    columns={f"YR{i}": f"YR{i}_exp" for i in range(2010, 2025)}, inplace=True
)
df_imp.rename(
    columns={f"YR{i}": f"YR{i}_imp" for i in range(2010, 2025)}, inplace=True
)


In [5]:
df_full = pd.merge(
    df_exp,
    df_imp,
    on=["economy", "Country"],
    how="outer",
)

In [6]:
for i in range(2010, 2025):
    df_full[f"YR{i}_balance"] = (
        df_full[f"YR{i}_exp"] - df_full[f"YR{i}_imp"]
    )
df_balance = df_full.drop(
    columns=[f"YR{i}_exp" for i in range(2010, 2025)]
    + [f"YR{i}_imp" for i in range(2010, 2025)]
)

In [7]:
df_balance_norm = df_balance.copy()
for i in range(2010, 2025):
    df_balance_norm[f"YR{i}_balance"] = (
        df_balance[f"YR{i}_balance"] * 100 / PIB_world.loc["WLD"][f"YR{i}"]
    )

In [8]:
df_balance_norm["YR2024_balance"].sum()

np.float64(0.7729117200643905)

In [9]:
df_balance_norm.to_csv(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/df_balance_norm.csv", index=False
)

In [10]:
df_mapping = pd.read_excel(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/Aggregation111Power_trump.xlsx",
    sheet_name="Mapping_table_country",
    header=1,
)

In [11]:
df_balance_norm_MIRAGE = pd.merge(
    left=df_balance_norm,
    right=df_mapping[["Code", "Aggregation Code"]],
    left_on="economy",
    right_on="Code",
    how="left",
)
df_balance_norm_MIRAGE["Aggregation Code"] = df_balance_norm_MIRAGE["Aggregation Code"].fillna("RoW")

In [12]:
df_balance_norm_MIRAGE


,Country,YR2010_balance,YR2011_balance,YR2012_balance,YR2013_balance,YR2014_balance,YR2015_balance,YR2016_balance,YR2017_balance,YR2018_balance,YR2019_balance,YR2020_balance,YR2021_balance,YR2022_balance,YR2023_balance,YR2024_balance,Code,Aggregation Code
0,Aruba,-0.000551,-0.000530,-0.000359,-0.000299,-0.000194,0.000006,0.000040,0.000021,-0.000075,-0.000170,-0.000445,-0.000142,0.000030,0.000209,0.000291,NaN,RoW
1,Afghanistan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.005976,-0.003281,-0.005091,-0.005517,-0.008336,AFG,RoW
2,Angola,0.023983,0.032412,0.034196,0.024972,0.008507,-0.003626,0.001789,0.005395,0.013918,0.012371,0.006000,0.013992,0.019309,0.011589,0.010994,NaN,RoW
3,Albania,-0.003670,-0.003940,-0.003019,-0.002949,-0.003129,-0.002592,-0.002593,-0.002387,-0.002372,-0.002371,-0.002547,-0.002432,-0.001905,-0.001207,-0.001667,ALB,RoW
4,Andorra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RoW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,Kosovo,-0.003103,-0.003090,-0.002619,-0.002507,-0.002647,-0.002417,-0.002372,-0.002252,-0.002545,-0.002413,-0.002885,-0.003032,-0.002984,-0.002990,-0.003039,NaN,RoW
213,"Yemen, Rep.",-0.002031,-0.001046,-0.005415,-0.004029,-0.002842,-0.006864,-0.008720,-0.007713,-0.009544,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RoW
214,South Africa,0.007242,0.004681,-0.007450,-0.010854,-0.007118,-0.005940,0.001893,0.005316,0.002360,0.002220,0.017183,0.026439,0.008447,0.002941,0.006958,ZAF,SSA
215,Zambia,0.001863,0.002569,0.001609,0.000183,0.000491,-0.001566,-0.000900,-0.000504,0.000309,0.000126,0.002988,0.004077,0.003144,0.000886,0.001683,ZMB,SSA


In [13]:
df_balance_norm_MIRAGE_grouped =df_balance_norm_MIRAGE.drop(columns=["Country", "Code"]).groupby(
    "Aggregation Code"
).sum()

In [14]:
df_balance_norm_MIRAGE_grouped.to_csv(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/df_balance_norm_MIRAGE.csv",
    index=False,
)